# Tetris Reinforcement Learning Project

**Ozan Kan - Northwestern University** 

**Objective:** Train agents to play Tetris using reinforcement learning. We compare two approaches: a tabular TD(0) value-learning agent (simple, interpretable) and a DQN-style neural network agent (stronger performance).

**Methodology:** We use a custom 20x10 Tetris engine with standard tetrominoes. The board is compressed into 4 scalar features to make the state space tractable. Each agent learns to choose the best piece placement (rotation + column) at each step by maximizing expected cumulative reward.

**Python environment only**. Requires: numpy, PyTorch, tetris_rl package (TetrisEngine + get_features).

**Overview:**
- **Environment**: Standard tetrominoes (I, O, T, S, Z, J, L). Each step: choose placement, receive reward, receive next piece.
- **State representation**: Board summarized into 4 features (aggregate height, holes, bumpiness, max height).
- **Agents**: Tabular (discretized state + TD(0) Bellman updates) and DQN (value network + replay buffer + target network).

In [11]:
# Setup: add project src to path; import only env and features (models/agents defined below)
import sys
from pathlib import Path
project_root = Path(".").resolve()
sys.path.insert(0, str(project_root / "src"))

import numpy as np
import random
from collections import defaultdict, deque
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tetris_rl.features import get_features
# TetrisEngine defined in notebook below (Section 1)

## 1. Tetris Environment

The `TetrisEngine` provides a 20x10 board and standard tetrominoes (I, O, T, S, Z, J, L). Key design: instead of stepping one move at a time, we use `get_next_states()`, which returns all valid placements for the current piece as a dictionary mapping `(rotation, column)` to `(board, reward, game_over)`. This lets the agent evaluate all options and pick the best before executing.

**Flow:** Agent calls `get_next_states()`, chooses action `(rot, x)`, calls `step(action)`, then the engine applies the placement, clears lines, assigns reward, and draws the next piece. If the agent attempts an illegal move, the game ends with reward -10.

**Reward scheme:**
- +1 per piece placed (survival bonus)
- +10 * (lines_cleared)^2 for line clears (incentivizes combos)
- -25 on game over (moderate penalty so one loss does not dominate learning; -100 was too harsh in early experiments)

In [12]:
# TetrisEngine (from tetris_rl.environment)
BOARD_HEIGHT, BOARD_WIDTH = 20, 10
TETROMINOS = {
    'I': [[(0,-1),(0,0),(0,1),(0,2)], [(-1,0),(0,0),(1,0),(2,0)]],
    'O': [[(0,0),(0,1),(1,0),(1,1)]],
    'T': [[(0,-1),(0,0),(0,1),(-1,0)], [(-1,0),(0,0),(1,0),(0,1)], [(0,-1),(0,0),(0,1),(1,0)], [(0,0),(-1,0),(0,1),(1,0)]],
    'S': [[(0,-1),(0,0),(-1,0),(-1,1)], [(-1,0),(0,0),(0,1),(1,1)]],
    'Z': [[(0,1),(0,0),(-1,0),(-1,-1)], [(-1,1),(0,1),(0,0),(1,0)]],
    'J': [[(0,-1),(0,0),(0,1),(-1,-1)], [(-1,0),(0,0),(1,0),(-1,1)], [(0,-1),(0,0),(0,1),(1,1)], [(1,-1),(-1,0),(0,0),(1,0)]],
    'L': [[(0,-1),(0,0),(0,1),(-1,1)], [(-1,0),(0,0),(1,0),(1,1)], [(0,-1),(0,0),(0,1),(1,-1)], [(-1,-1),(-1,0),(0,0),(1,0)]]
}

class TetrisEngine:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = np.zeros((BOARD_HEIGHT, BOARD_WIDTH), dtype=int)
        self.score, self.game_over = 0, False
        self.current_piece = self.get_new_piece()
        return self.board

    def get_new_piece(self):
        name = random.choice(list(TETROMINOS.keys()))
        return {'name': name, 'rotations': TETROMINOS[name]}

    def is_valid_position(self, board, piece_coords, offset_y, offset_x):
        for (y, x) in piece_coords:
            r, c = offset_y + y, offset_x + x
            if c < 0 or c >= BOARD_WIDTH or r >= BOARD_HEIGHT:
                return False
            if r >= 0 and board[r, c] == 1:
                return False
        return True

    def get_next_states(self):
        states = {}
        for rot_idx, shape_coords in enumerate(self.current_piece['rotations']):
            for x in range(-2, BOARD_WIDTH + 2):
                if not self.is_valid_position(self.board, shape_coords, 0, x):
                    continue
                y = 0
                while self.is_valid_position(self.board, shape_coords, y + 1, x):
                    y += 1
                next_board = self.board.copy()
                valid = True
                for (py, px) in shape_coords:
                    r, c = y + py, x + px
                    if 0 <= r < BOARD_HEIGHT and 0 <= c < BOARD_WIDTH:
                        next_board[r, c] = 1
                    else:
                        valid = False
                if not valid:
                    continue
                cleared_board, lines = self.clear_lines(next_board)
                reward = 1.0 + (lines ** 2) * 10
                is_game_over = np.any(cleared_board[0, :] == 1)
                if is_game_over:
                    reward -= 25
                states[(rot_idx, x)] = (cleared_board, reward, is_game_over)
        return states

    def step(self, action):
        rot_idx, x = action
        possible_states = self.get_next_states()
        if (rot_idx, x) in possible_states:
            self.board, reward, self.game_over = possible_states[(rot_idx, x)]
            self.score += reward
            self.current_piece = self.get_new_piece()
            return reward, self.game_over
        return -10, True

    def clear_lines(self, board):
        full_rows = np.all(board == 1, axis=1)
        num_cleared = np.sum(full_rows)
        if num_cleared > 0:
            board = board[~full_rows]
            board = np.vstack((np.zeros((num_cleared, BOARD_WIDTH), dtype=int), board))
        return board, num_cleared

In [ ]:
# Demo: create env, inspect possible moves, run one step
env = TetrisEngine()
board = env.reset()
print("First piece:", env.current_piece["name"])
next_states = env.get_next_states()
print(f"Possible moves: {len(next_states)}")
action = list(next_states.keys())[0]
reward, done = env.step(action)
print(f"After one step: reward={reward}, game_over={done}")

## 2. State Representation (Features)

The raw 20x10 board is too large for tabular methods. We reduce it to 4 scalar features:

1. **Aggregate height**: Sum of column heights. Higher values mean the stack is filling up.
2. **Holes**: Count of empty cells below blocks. Holes make future line clears harder.
3. **Bumpiness**: Sum of |height[i] - height[i+1]| across columns. Smoother surfaces are easier to clear.
4. **Max height**: Highest column. Critical for avoiding game over.

`get_features(board)` returns `[agg_height, holes, bumpiness, max_height]`. These features are computed from column heights and hole counts; the implementation is in `tetris_rl.features` (get_column_heights, get_holes, get_bumpiness).

In [14]:
# Feature extraction (from tetris_rl.features - shown for completeness)
def get_column_heights(board):
    heights = []
    for i in range(board.shape[1]):
        if np.any(board[:, i]):
            top = np.argmax(board[:, i])
            heights.append(20 - top)
        else:
            heights.append(0)
    return heights

def get_holes(board):
    total = 0
    for c in range(board.shape[1]):
        col = board[:, c]
        if np.any(col):
            argmax = np.argmax(col)
            total += np.count_nonzero(col[argmax:] == 0)
    return total

def get_bumpiness(board):
    h = get_column_heights(board)
    return sum(abs(h[i] - h[i+1]) for i in range(len(h)-1))

In [ ]:
# Demo: features for current board
env = TetrisEngine()
board = env.reset()
f = get_features(board)
print("Features [agg_height, holes, bumpiness, max_height]:", f)

## 3. Tabular TD(0) Agent

We maintain a value table V(s) over discretized states. Each 4-tuple of features is bucketed into integers (e.g., agg_height/15, holes capped at 7) to form a state key. Actions are scored as `reward + gamma * V(next_state)`; we pick the action with the highest score (epsilon-greedy for exploration).

**TD(0) update:** After taking action and observing reward r and next state s', we update:  
`V(s) <- V(s) + lr * (r + gamma*V(s') - V(s))`  
This bootstraps from V(s') instead of waiting for the full return. Hyperparameters: lr=0.2, gamma=0.98, epsilon=0.5 (decayed over training). The agent learns slowly because discretization loses detail; finer buckets help but increase the state space.

In [16]:
# TabularAgent: discretize features, epsilon-greedy, TD(0) update
class TabularAgent:
    def __init__(self):
        self.q_table = defaultdict(float)
        self.learning_rate = 0.2
        self.gamma = 0.98
        self.epsilon = 0.5

    def discretize(self, features):
        agg_height, holes, bumpiness, max_height = features[0], features[1], features[2], features[3]
        agg_bucket = min(7, int(agg_height / 15))
        holes_bucket = min(7, int(holes))
        bump_bucket = min(6, int(bumpiness / 4))
        max_h_bucket = min(7, int(max_height / 3))
        return (agg_bucket, holes_bucket, bump_bucket, max_h_bucket)

    def select_action(self, next_states):
        if random.random() < self.epsilon:
            return random.choice(list(next_states.keys()))
        best_score, best_action = -float('inf'), None
        for action, (board, reward, game_over) in next_states.items():
            features = get_features(board)
            buckets = self.discretize(features)
            score = reward + self.gamma * self.q_table.get(buckets, 0.0)
            if score > best_score:
                best_score, best_action = score, action
        return best_action

    def update(self, current_features, reward, next_state_features, game_over):
        feature_buckets = self.discretize(current_features)
        value = self.q_table.get(feature_buckets, 0.0)
        next_value = 0.0 if game_over else self.q_table.get(self.discretize(next_state_features), 0.0)
        td_target = reward + self.gamma * next_value
        self.q_table[feature_buckets] = value + self.learning_rate * (td_target - value)

## 4. DQN Agent

We approximate V(s) with a neural network instead of a table. This allows generalization across similar states.

**Architecture:** 4 inputs (features) -> 64 -> 64 -> 64 -> 1. ReLU activations, orthogonal initialization (to avoid early value explosion). The output is the estimated state value.

**Replay buffer:** We store (s, r, s', done) transitions and sample random batches for training. This breaks temporal correlation and stabilizes learning.

**Target network:** A frozen copy of the main network is used to compute TD targets. We sync it every 500 steps. This prevents the target from moving too quickly and reduces instability.

**Action selection:** Same as tabular: argmax over `reward + gamma*V(s')` for each candidate next state. Epsilon-greedy (epsilon decays 0.995 per episode) for exploration.

In [17]:
# DQNModel: 4 -> 64 -> 64 -> 64 -> 1, ReLU, orthogonal init
INPUT_SIZE = 4
class DQNModel(nn.Module):
    def __init__(self, hidden_layer_size=64):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_SIZE, hidden_layer_size)
        self.fc2 = nn.Linear(hidden_layer_size, hidden_layer_size)
        self.fc3 = nn.Linear(hidden_layer_size, hidden_layer_size)
        self.out = nn.Linear(hidden_layer_size, 1)
        self._init_weights()
    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.orthogonal_(layer.weight, gain=nn.init.calculate_gain("relu"))
            nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.out.weight, gain=0.01)
        nn.init.zeros_(self.out.bias)
    def forward(self, features):
        if features.dim() == 1:
            features = features.unsqueeze(0)
        x = F.relu(self.fc1(features))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.out(x).squeeze(-1)

In [18]:
# ReplayBuffer + DQNAgent
class ReplayBuffer:
    def __init__(self, queue_len):
        self.memqueue = deque(maxlen=queue_len)
    def save(self, s, r, s_next, done):
        self.memqueue.append((s, r, s_next, done))
    def recall(self, batch_size=64):
        exps = random.sample(self.memqueue, k=batch_size)
        states_t = torch.tensor(np.array([e[0] for e in exps]), dtype=torch.float32)
        rewards_t = torch.tensor([e[1] for e in exps], dtype=torch.float32)
        next_states_t = torch.tensor(np.array([e[2] for e in exps]), dtype=torch.float32)
        game_overs_t = torch.tensor([e[3] for e in exps], dtype=torch.float32)
        return states_t, rewards_t, next_states_t, game_overs_t
    def size(self):
        return len(self.memqueue)

class DQNAgent:
    def __init__(self, batch_size=64, queue_len=100000, hidden_layer_size=64):
        self.gamma, self.epsilon, self.epsilon_min = 0.98, 0.5, 0.01
        self.epsilon_decay, self.batch_size = 0.995, batch_size
        self.buffer = ReplayBuffer(queue_len)
        self.model = DQNModel(hidden_layer_size)
        self.target_model = DQNModel(hidden_layer_size)
        self.target_model.load_state_dict(self.model.state_dict())
        self.target_update_freq, self.learn_steps = 500, 0
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-3)

    def predict_value(self, features):
        with torch.no_grad():
            return self.model(torch.tensor(features, dtype=torch.float32)).item()

    def act(self, next_states):
        if random.random() < self.epsilon:
            return random.choice(list(next_states.keys()))
        best_score, best_action = -float('inf'), None
        for action, (board, reward, game_over) in next_states.items():
            score = reward if game_over else reward + self.gamma * self.predict_value(get_features(board))
            if score > best_score:
                best_score, best_action = score, action
        return best_action

    def learn(self):
        if self.buffer.size() < self.batch_size:
            return
        states, rewards, next_states, game_overs = self.buffer.recall(self.batch_size)
        with torch.no_grad():
            targets = rewards + self.gamma * self.target_model(next_states) * (1.0 - game_overs)
        loss = nn.MSELoss()(self.model(states), targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.learn_steps += 1
        if self.learn_steps % self.target_update_freq == 0:
            self.target_model.load_state_dict(self.model.state_dict())

    def update_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

## 5. Training

**Episode loop:** Each episode runs until game over. For each step: (1) call `get_next_states()` to get all valid placements, (2) agent selects action (rotation, column), (3) call `env.step(action)`, (4) agent updates. Tabular: TD(0) update on (state_before, reward, state_after). DQN: store (s, r, s', done) in buffer, then sample batch and perform gradient step with MSE loss vs TD target.

**Tabular specifics:** Epsilon and learning rate decay over time (0.9997 and 0.99995 per episode) so the agent explores less and makes smaller updates as it converges. We log rolling 100-episode average score (Avg100) and number of unique states visited.

**DQN specifics:** `agent.learn()` samples 64 transitions, computes targets with the target network, updates the main network. Epsilon decays 0.995 per episode. We cap pieces per game at 5000 to avoid rare infinite runs. Below: 500-episode demo; full runs use 10k episodes.

In [ ]:
# Tabular training (exact code from train_tabular.py)
def train_tabular(episodes=10000):
    env = TetrisEngine()
    agent = TabularAgent()
    score_window = []
    for episode in range(episodes):
        board = env.reset()
        game_over = False
        current_features = get_features(board)
        while not game_over:
            state_before = get_features(env.board)
            possible_moves = env.get_next_states()
            if not possible_moves:
                break
            best_action = agent.select_action(possible_moves)
            reward, game_over = env.step(best_action)
            state_after = get_features(env.board)
            agent.update(state_before, reward, state_after, game_over)
            current_features = state_after
        score_window.append(env.score)
        if len(score_window) > 100:
            score_window.pop(0)
        agent.epsilon = max(0.02, agent.epsilon * 0.9997)
        agent.learning_rate = max(0.02, agent.learning_rate * 0.99995)
        if (episode + 1) % 100 == 0:
            avg = sum(score_window) / len(score_window)
            print(f"Episode: {episode + 1} | Score: {env.score} | Avg100: {avg:.1f} | Epsilon: {agent.epsilon:.3f} | States: {len(agent.q_table)}")

train_tabular(episodes=500)

In [ ]:
# DQN training (exact code from train_dqn_py.py)
MAX_PIECES_IN_GAME = 5000

def train_dqn(batch_size=64, queue_len=100000, hidden_layer_size=64, episodes=10000):
    env = TetrisEngine()
    agent = DQNAgent(batch_size, queue_len, hidden_layer_size)
    score_window = []
    for episode in range(episodes):
        board = env.reset()
        game_over = False
        pieces = 0
        while not game_over:
            state_before = get_features(env.board)
            possible_moves = env.get_next_states()
            if not possible_moves:
                break
            best_action = agent.act(possible_moves)
            reward, game_over = env.step(best_action)
            state_after = get_features(env.board)
            agent.buffer.save(state_before, reward, state_after, game_over)
            agent.learn()
            pieces += 1
            if pieces > MAX_PIECES_IN_GAME:
                game_over = True
        score_window.append(env.score)
        if len(score_window) > 100:
            score_window.pop(0)
        agent.update_epsilon()
        if (episode + 1) % 100 == 0:
            avg = sum(score_window) / len(score_window)
            print(f"Episode: {episode + 1} | Score: {env.score} | Avg100: {avg:.1f} | Epsilon: {agent.epsilon:.3f} | Buffer: {agent.buffer.size()}")

train_dqn(episodes=500)

## 6. Results (Full 10k-Episode Runs)

| Agent | Final Avg100 | Peak Score | Time |
|-------|--------------|------------|------|
| Tabular | ~163 | 408 | ~10 min |
| DQN (Python) | ~2,900 | 11,489 | ~1 hr |

**Tabular:** Learns survival but plateaus around Avg100 ~160. Coarse discretization loses detail; many boards map to the same state. ~500 unique states visited.

**DQN:** Much stronger. The network generalizes across features; Avg100 reaches ~3000+ within 2k episodes. Replay buffer and target network stabilize training. Peaks over 11k show long sustained games.

**Conclusion:** Function approximation (DQN) significantly outperforms tabular for this problem. The 4-feature representation works for both; the tabular bottleneck is discretization granularity.

## 7. Credits

This file only contains the essential logic and code for the Tetris RL project, and some functionality is omitted. If you want to get more info and take a more detailed look at the whole codebase, including detailed outputs for training phases, please visit the public repo at https://github.com/ozikan5/Tetris-RL-Agent.